In [2]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import os

ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src import setup_workspace, query_to_df, ds3on1

# 1. Inicializar workspace y rutas
paths = setup_workspace()

## 2. Consultar directamente a PostgreSQL (vía DuckDB ATTACH)
#df = query_to_df("SELECT * FROM db.EuInfo;")

# 3. Análisis univariable 3-en-1
# ds3on1(df, 'happy', Desc=True, Stat=True, Graph=True)

✅ Workspace anclado en: /home/josu/Documentos/DataScienceCourse
📂 Ruta de tablas verificada: /home/josu/Documentos/DataScienceCourse/data/raw/Tables


🟢 < Module 9.1 > Predictive Modeling, Ablation Study & Country-Grouped Cross-Validation (GroupKFold):

##### 🎯 Context & Objectives

In hierarchical socio-economic datasets—where individual survey respondents (ESS) are nested within countries and macroeconomic clusters (Eurostat)—standard random cross-validation (`KFold` or `StratifiedKFold`) violates the independent and identically distributed (**i.i.d.**) assumption[cite: 3, 4, 5]. Mixing citizens from the same country across training and validation folds allows models to memorize national baselines ("country leakage"), yielding artificially inflated evaluation scores[cite: 4, 5].

In this module, you will build a leakage-free predictive modeling pipeline using **country-grouped cross-validation (`GroupKFold`)**[cite: 3, 4, 5] and execute a controlled **Feature Ablation Study**[cite: 3]. By keeping the algorithm and cross-validation splits fixed while varying the input feature blocks (Individual Microdata vs. Macroeconomic Clusters vs. Combined Hybrid), you will scientifically test whether macro-regional economic indicators provide statistically meaningful predictive power on unseen countries[cite: 3, 4, 5].

---

##### 📑 Exercise Instructions

###### 1. Syntactic / Algorithmic Level (Leakage-Free GroupKFold Pipeline)

Create a Python notebook (`notebooks/09_1_predictive_modeling_groupkfold.ipynb`) that implements the following architectural steps:

- **SQL Ingestion & Clean Data Retrieval:**
    - Connect to your local PostgreSQL instance via DuckDB (`src/db.py`) and load the cleaned survey microdata merged with the macroeconomic country features (`cntry`, individual attitudes/demographics, and macro indicators/clusters)[cite: 3, 8].
    - Ensure raw rows are used prior to global oversampling so that `GroupKFold` can isolate complete countries (`groups = df['cntry']`) without synthetic cross-border contamination[cite: 4, 5].

- **Inside-the-Fold Asymmetric Preprocessing (`imblearn.pipeline.Pipeline`):**
    - Configure `GroupKFold(n_splits=5)` using `cntry` as the grouping vector[cite: 3, 4, 5].
    - Encapsulate your preprocessing transformations (`StandardScaler`, `PCA(n_components=1)` on institutional trust, and calibrated `SMOTE(random_state=42)` + post-synthesis discrete rounding) **strictly inside each training fold** so that validation folds remain 100% untouched real respondents.

---

###### 2. Applied / Data Level (Controlled Ablation Study & Bayesian Baseline)

Evaluate whether macroeconomic context improves out-of-country generalization compared to individual-level predictors alone[cite: 3, 4]:

- **Four-Tier Feature Ablation Experiment:**
    - Train and evaluate the same baseline classifier (e.g., `LogisticRegression` or `HistGradientBoostingClassifier`) across four controlled feature configurations under identical `GroupKFold` splits[cite: 3, 4, 5]:
        1. **Tier 0 — Null Baseline (`DummyClassifier`):** Majority-class baseline establishing the random-guess floor.
        2. **Tier 1 — Individual Microdata Only ($X_{\text{micro}}$):** Only respondent-level variables (`lrscale`, `institutional_trust_index`, `n_nonresp`, `has_nonresp`, demographics).
        3. **Tier 2 — Macroeconomic Context Only ($X_{\text{macro}}$):** Only country-level macroeconomic metrics and structural clusters (GDP, inflation, unemployment, economic cluster dummies)[cite: 3].
        4. **Tier 3 — Hybrid Model ($X_{\text{micro}} + X_{\text{macro}}$):** Full feature matrix combining individual microdata and macroeconomic context[cite: 3].

- **Probabilistic / Bayesian Comparison & Error Analysis:**
    - Compare your discriminative classifier against a probabilistic Bayesian baseline (`GaussianNB` or Bayesian Logistic Regression) on the Hybrid feature set.
    - Report **ROC-AUC**, **Macro F1-Score**, and **Balanced Accuracy** (mean $\pm$ std across the 5 country folds) in a summary comparison table.
    - **Conceptual Conclusion:** Based on the delta between Tier 1 ($X_{\text{micro}}$) and Tier 3 ($X_{\text{hybrid}}$), state formally whether you reject or fail to reject the null hypothesis ($H_0$) regarding the predictive power of macroeconomic clusters on unseen countries, discussing the role of unobserved individual latent factors.

---

##### 🌐 Official References & Documentation

- **Scikit-Learn GroupKFold:** [GroupKFold Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html)
- **Scikit-Learn Cross-Validation on Grouped Data:** [Group CV Guide](https://scikit-learn.org/stable/modules/cross_validation.html#group-cv)
- **Imbalanced-Learn Pipeline:** [imblearn.pipeline.Pipeline Documentation](https://imbalanced-learn.org/stable/references/generated/imblearn.pipeline.Pipeline.html)

---

##### 📚 Recommended Manuals & CheatSheets

- **Multilevel & Hierarchical Modeling:** *Data Analysis Using Regression and Multilevel/Hierarchical Models (Andrew Gelman & Jennifer Hill)* — Chapter 11–12 (Nested/Clustered Survey Data).
- **Model Evaluation & Ablation:** *Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow (Aurélien Géron)* — Chapter 3 (Classification Metrics: ROC-AUC vs. F1).

💻 **Your Challenge:** Create `notebooks/09_1_predictive_modeling_groupkfold.ipynb` (or `notebooks/Ex9.1.ipynb`), implement the 5-fold `GroupKFold` pipeline with inside-the-fold `SMOTE`, run the 4-tier ablation comparison (`Dummy` vs. `Micro` vs. `Macro` vs. `Hybrid` + `GaussianNB`), and share your cross-validation metrics table to audit the scientific conclusion! 🚀📊

In [3]:
# Step 1: Lets create our study table, we must stratify the sample to not lose or intoxicate the statistics shape.
# We must merge using key value from 'cntry' column of "ess_ex81_smote_split" table, and key values from 'geo' column of "macro_panel" table, also adding new category "Sinthetic from SMOTE samples"
# We must use random seed for determinism results
import pandas as db
import numpy as np
import pandas as pd
import random


# Extract the stratified samples between "split_set"-"happy_high" values with the macroeconomic clustering.
# We will select 1k samples per combination.
df91 = query_to_df("""
    WITH joined_data AS (
        SELECT 
            a.cntry,
            a.split_set,
            a.lrscale,
            a.n_nonresp,
            a.has_nonresp,
            a.institutional_trust_index,
            a.happy_high,
            b.* EXCLUDE (geo),
            ROW_NUMBER() OVER (
                PARTITION BY a.split_set, a.happy_high 
                ORDER BY hash(a.cntry, a.lrscale, a.institutional_trust_index, 42)
            ) AS rn
        FROM db.ess_ex81_smote_split a
        INNER JOIN db.macro_panel b 
            ON a.cntry = b.geo
        WHERE a.cntry <> 'SYNTHETIC'
        AND b.year == '2024'
    )
    SELECT * EXCLUDE (rn)
    FROM joined_data
    WHERE rn <= 1000;
""")

print(f"✅ Stratified Dataset: {df91.shape}")
print(df91.groupby(["split_set", "happy_high"]).size())

✅ Stratified Dataset: (4000, 14)
split_set  happy_high
test       0             1000
           1             1000
train      0             1000
           1             1000
dtype: int64


In [12]:
#Lets check how many different values, check the list of unique values, and the frecuency of each one.
print(df91['cntry'].nunique()) 
print(df91['cntry'].unique())         
print(df91['cntry'].value_counts())

15
<ArrowStringArray>
['FI', 'HU', 'GB', 'PT', 'EE', 'BE', 'SE', 'NO', 'IE', 'ES', 'DE', 'AT', 'PL',
 'FR', 'NL']
Length: 15, dtype: str
cntry
AT    621
FR    563
PL    458
NL    358
IE    270
HU    241
PT    203
DE    202
GB    199
BE    171
FI    163
ES    157
EE    149
SE    125
NO    120
Name: count, dtype: int64


In [ ]:
#Remember that our macroregion and economic clustering doesnt have all european countries, just the most relevant for our study.
